
# d+Au @ 200 GeV Glauber (Optical + Monte Carlo)

This notebook imports your uploaded `glauber.py` and computes Glauber geometry for **d+Au at $$\sqrt{s_{NN}}=200$$ GeV**:

- **Min-bias** $$\langle b\rangle$$ for **0–100%** inelastic and **0–88%** (PHENIX MB-trigger-like)
- Centrality-bin tables for **0–20–40–60–88%** (+ 88–100% and 0–100% reference)
- $$\langle N_{\mathrm{coll}}\rangle$$, $$\langle N_{\mathrm{part}}\rangle$$, and optical splits $$N_{\mathrm{part}}[\mathrm{Au}], N_{\mathrm{part}}[d]$$
- Plots: $$b$$ distribution (MC vs optical inelastic weight), $$N_{\mathrm{coll}}$$ distributions by bin (log y)

## Parameter choices (PHENIX-style)

- Au Woods–Saxon: $$R=6.38\ \mathrm{fm},\ a=0.54\ \mathrm{fm}$$  
- Deuteron Hulthén: $$\alpha=0.228\ \mathrm{fm}^{-1},\ \beta=1.18\ \mathrm{fm}^{-1}$$  
- $$\sigma_{NN}^{\mathrm{inel}}=42\ \mathrm{mb}$$ at 200 GeV

## Optical inelastic weighting

$$
T_{dA}(b)=\int d^2s\, T_d(\mathbf{s})\,T_A(\mathbf{s}-\mathbf{b}),
\qquad
P_{\mathrm{inel}}(b)=1-\exp\left[-\sigma_{NN}\,T_{dA}(b)\right]
$$

$$
\frac{d\sigma_{dA}^{\mathrm{inel}}}{db} = 2\pi b\,P_{\mathrm{inel}}(b),
\qquad
\langle X\rangle_{\mathrm{MB}}=
\frac{\int db\; 2\pi b\,P_{\mathrm{inel}}(b)\,X(b)}
{\int db\; 2\pi b\,P_{\mathrm{inel}}(b)}.
$$

> Centrality in this notebook is defined by **$$N_{\mathrm{coll}}$$ quantiles** (a clean proxy).  
> To reproduce PHENIX exactly, add Glauber+NBD+BBC response (can be a follow-up).


In [ ]:

# Imports + load your uploaded glauber.py
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys, importlib.util, importlib.machinery
glauber_path = r"/mnt/data/glauber.py"
loader = importlib.machinery.SourceFileLoader("glauber_user", glauber_path)
spec = importlib.util.spec_from_loader(loader.name, loader)
glauber = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = glauber
loader.exec_module(glauber)

print("Loaded:", glauber_path)


In [ ]:

# --- Parameters (PHENIX-style defaults for d+Au @ 200 GeV) ---
ROOTS = 200.0
A_AU  = 197

SIGMA_NN_MB = 42.0   # mb
AU_A_FM     = 0.54   # diffuseness (fm)

HUL_ALPHA   = 0.228  # fm^{-1}
HUL_BETA    = 1.18   # fm^{-1}

CENT_EDGES = [0, 20, 40, 60, 88, 100]   # percent
BMAX_FM = 25.0
SEED = 7

# Increase N_EVENTS for stable numbers (start with 50k; 200k+ is nicer)
N_EVENTS = 50000


In [ ]:

# -----------------------------
# Monte Carlo Glauber (d+Au)
# -----------------------------
cfg = glauber.MCConfig(
    n_events=N_EVENTS,
    bmax_fm=BMAX_FM,
    sigma_nn_mb=SIGMA_NN_MB,
    A=A_AU,
    d_fm=AU_A_FM,
    seed=SEED,
    hulthen_alpha=HUL_ALPHA,
    hulthen_beta=HUL_BETA,
)
mc = glauber.MonteCarloGlauber(cfg)
out = mc.run(system="dA")

b = out["b"]
Ncoll = out["Ncoll"]
Npart = out["Npart"]

# inelastic events
inel = Ncoll > 0
b_inel = b[inel]
Ncoll_inel = Ncoll[inel]
Npart_inel = Npart[inel]

print("N_total =", len(b), "  N_inel =", len(b_inel))
print("<b> 0-100% inelastic =", b_inel.mean())

# 0-88% trigger-like proxy: keep most central 88% of inelastic events by Ncoll
order = np.argsort(Ncoll_inel)[::-1]
keep = order[: int(round(0.88*len(order))) ]
print("<b> 0-88% (Ncoll-proxy) =", b_inel[keep].mean())


In [ ]:

# -----------------------------
# Centrality table (MC, Ncoll-quantile proxy)
# -----------------------------
edges = CENT_EDGES
masks = glauber.MonteCarloGlauber.centrality_slices(Ncoll_inel, edges)
means = lambda arr: glauber.MonteCarloGlauber.mean_in_bins(arr, masks)

df_mc = pd.DataFrame({
    "centrality": [f"{edges[i]}-{edges[i+1]}%" for i in range(len(edges)-1)],
    "b_mean_fm": means(b_inel),
    "Ncoll_mean": means(Ncoll_inel),
    "Ncoll_min": [int(Ncoll_inel[m].min()) if np.any(m) else 0 for m in masks],
    "Ncoll_max": [int(Ncoll_inel[m].max()) if np.any(m) else 0 for m in masks],
    "Npart_mean": means(Npart_inel),
    "Npart_min": [int(Npart_inel[m].min()) if np.any(m) else 0 for m in masks],
    "Npart_max": [int(Npart_inel[m].max()) if np.any(m) else 0 for m in masks],
})
df_mc


In [ ]:

# -----------------------------
# Plot: Ncoll distributions per bin (log y)
# -----------------------------
plt.figure(figsize=(7,4))
for m, lab in zip(masks, df_mc["centrality"]):
    plt.hist(Ncoll_inel[m],
             bins=np.arange(0, int(Ncoll_inel.max())+2)-0.5,
             density=True, histtype="step", label=lab)
plt.yscale("log")
plt.xlabel("Ncoll")
plt.ylabel("Probability density")
plt.title("d+Au @ 200 GeV: Ncoll distributions by centrality (MC)")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()



# Optical Glauber (slower)

This runs a smooth optical calculation and computes $$\langle b\rangle$$ and bin-averages using:
$$
w(b)=2\pi b\left(1-e^{-\sigma_{NN}T_{dA}(b)}\right).
$$

If it runs too slowly on your laptop, reduce `nx`, `ny`, `nb` below.


In [ ]:

# -----------------------------
# Optical Glauber (d+Au)
# -----------------------------
spec_dAu = glauber.SystemSpec(system="dA", roots_GeV=ROOTS, A=A_AU,
                             sigma_nn_mb=SIGMA_NN_MB, diffuseness_fm=AU_A_FM)

opt = glauber.OpticalGlauber(
    spec_dAu,
    verbose=True,
    bmax_fm=BMAX_FM,
    nb=201,
    xylim_fm=20.0,
    nx=120, ny=120,
)

def w_inel_dA(b: np.ndarray) -> np.ndarray:
    T = np.interp(b, opt.b_grid, opt.TdA_b)
    lam = opt.sigma_nn_fm2 * np.maximum(T, 0.0)
    pinel = 1.0 - np.exp(-lam)
    return 2.0*math.pi*b*pinel

def mean_over_b(X_of_b, c0=0.0, c1=1.0, n=4000) -> float:
    bmin, bmax = opt._bin_edges_b(c0, c1, "dA")
    b = np.linspace(bmin, bmax, n)
    w = w_inel_dA(b)
    X = X_of_b(b)
    return float(np.trapz(X*w, b) / max(np.trapz(w, b), 1e-30))

print("Optical <b> 0-100% inelastic =", mean_over_b(lambda b: b, 0.0, 1.0))
print("Optical <b> 0-88% trigger-like =", mean_over_b(lambda b: b, 0.0, 0.88))
